In [1]:
from typing import Dict
import chromadb
import pandas as pd

def prepare_emission_documents(csv_path: str) -> Dict:
    """
    Convert emission activities CSV into ChromaDB-ready documents.
    Each emission activity becomes a searchable document.
    """
    df = pd.read_csv(csv_path)

    documents = []
    metadatas = []
    ids = []

    for index, row in df.iterrows():
        # Create rich document text for semantic search
        document_text = f"""
        Emission Activity: {row['EmissionSource']}
        Scope: {row['EmissionScope']}
        Category: {row['ActivityCategory']}
        
        Data Collection Requirements:
        - What to collect: {row['DataToCollect']}
        - Unit of measure: {row['UnitOfMeasure']}
        - Typical percentage of total footprint: {row['TypicalPercentage']}%
        - Priority level: {row['PriorityLevel']}
        
        This is a {row['EmissionScope']} emission source in the {row['ActivityCategory']} category.
        For Bitcoin mining operations, this activity typically represents {row['TypicalPercentage']}% of total emissions.
        Collection priority: {row['PriorityLevel']}.
        """.strip()

        # Parse percentage range to get min and max values
        percentage_str = str(row['TypicalPercentage'])
        if percentage_str == 'N/A':
            percentage_min = 0.0
            percentage_max = 0.0
            percentage_display = "N/A"
        else:
            # Handle ranges like "95-98" or single values
            if '-' in percentage_str:
                parts = percentage_str.split('-')
                percentage_min = float(parts[0])
                percentage_max = float(parts[1])
                percentage_display = percentage_str
            else:
                percentage_min = float(percentage_str)
                percentage_max = float(percentage_str)
                percentage_display = percentage_str

        # Rich metadata for filtering and exact lookups
        # ChromaDB only supports: str, int, float, bool
        metadata = {
            "emission_scope": row["EmissionScope"].lower(),
            "activity_category": row["ActivityCategory"].lower(),
            "emission_source": row["EmissionSource"].lower(),
            "data_to_collect": row["DataToCollect"].lower(),
            "unit_of_measure": row["UnitOfMeasure"].lower(),
            "percentage_range": percentage_display,  # String for display
            "percentage_min": percentage_min,  # Float for filtering
            "percentage_max": percentage_max,  # Float for filtering
            "priority_level": row["PriorityLevel"].lower(),
            # Add searchable keywords
            "keywords": f"{row['EmissionScope']} {row['ActivityCategory']} {row['EmissionSource']}".lower().replace(
                " ", "_"
            ),
            # Parse priority ranking
            "priority_rank": {
                "critical": 5,
                "high": 4,
                "medium": 3,
                "low-medium": 2,
                "low": 1
            }.get(row["PriorityLevel"].lower(), 0)
        }

        documents.append(document_text)
        metadatas.append(metadata)
        ids.append(f"emission_{row['EmissionScope']}_{index}")

    return {"documents": documents, "metadatas": metadatas, "ids": ids}

In [3]:
def setup_emission_chromadb(csv_path: str, collection_name: str = "bitcoin_mining_emissions"):
    """
    Create and populate ChromaDB collection with Bitcoin mining emission data.
    """
    # Initialize ChromaDB
    client = chromadb.PersistentClient("./chroma")

    # Create collection (delete if exists)
    try:
        client.delete_collection(collection_name)
    except BaseException:
        pass

    collection = client.create_collection(
        name=collection_name,
        metadata={
            "description": "Bitcoin mining emission tracking activities for Scope 1 and 2",
            "industry": "cryptocurrency_mining",
            "framework": "GHG_Protocol"
        },
    )

    # Prepare documents
    data = prepare_emission_documents(csv_path)

    # Add to ChromaDB
    collection.add(
        documents=data["documents"], 
        metadatas=data["metadatas"], 
        ids=data["ids"]
    )

    print(
        f"Added {len(data['documents'])} emission activities to ChromaDB collection '{collection_name}'"
    )
    return collection

In [4]:
# Create the collection
collection = setup_emission_chromadb("data/bitcoin_mining_emissions.csv")

Added 21 emission activities to ChromaDB collection 'bitcoin_mining_emissions'


In [5]:
# Connect to existing collection
chroma_client = chromadb.PersistentClient(path="./chroma")
emission_db = chroma_client.get_collection(name="bitcoin_mining_emissions")

In [6]:
print("=" * 80)
print("QUERY 1: Find all Scope 2 activities")
print("=" * 80)
results = emission_db.query(
    query_texts=["Scope 2 electricity consumption"],
    n_results=5,
    where={"emission_scope": "scope2"}
)
for i, doc in enumerate(results["documents"][0]):
    print(f"\n--- Result {i+1} ---")
    print(f"Priority: {results['metadatas'][0][i]['priority_level']}")
    print(f"Source: {results['metadatas'][0][i]['emission_source']}")
    print(f"Percentage Range: {results['metadatas'][0][i]['percentage_range']}")
    print(f"Priority Rank: {results['metadatas'][0][i]['priority_rank']}")
    print(f"\n{doc}")

QUERY 1: Find all Scope 2 activities

--- Result 1 ---
Priority: medium
Source: administrative office electricity
Percentage Range: 0.05-0.2
Priority Rank: 3

Emission Activity: Administrative office electricity
        Scope: Scope2
        Category: PurchasedElectricity

        Data Collection Requirements:
        - What to collect: Electricity usage
        - Unit of measure: kWh
        - Typical percentage of total footprint: 0.05-0.2%
        - Priority level: MEDIUM

        This is a Scope2 emission source in the PurchasedElectricity category.
        For Bitcoin mining operations, this activity typically represents 0.05-0.2% of total emissions.
        Collection priority: MEDIUM.

--- Result 2 ---
Priority: high
Source: facility operations electricity
Percentage Range: 0.1-0.5
Priority Rank: 4

Emission Activity: Facility operations electricity
        Scope: Scope2
        Category: PurchasedElectricity

        Data Collection Requirements:
        - What to collect: Elec

In [6]:
print("\n" + "=" * 80)
print("QUERY 2: Find critical priority activities")
print("=" * 80)
results = emission_db.query(
    query_texts=["most important emission sources"],
    n_results=5,
    where={"priority_level": "critical"}
)
for i, doc in enumerate(results["documents"][0]):
    print(f"\n--- Critical Activity {i+1} ---")
    print(f"Scope: {results['metadatas'][0][i]['emission_scope']}")
    print(f"Category: {results['metadatas'][0][i]['activity_category']}")
    print(f"Percentage Range: {results['metadatas'][0][i]['percentage_range']}")
    print(f"\n{doc}")


QUERY 2: Find critical priority activities

--- Critical Activity 1 ---
Scope: scope2
Category: purchasedelectricity
Percentage Range: 1-3

Emission Activity: Cooling system electricity
        Scope: Scope2
        Category: PurchasedElectricity

        Data Collection Requirements:
        - What to collect: Electricity usage
        - Unit of measure: kWh
        - Typical percentage of total footprint: 1-3%
        - Priority level: CRITICAL

        This is a Scope2 emission source in the PurchasedElectricity category.
        For Bitcoin mining operations, this activity typically represents 1-3% of total emissions.
        Collection priority: CRITICAL.

--- Critical Activity 2 ---
Scope: scope2
Category: purchasedelectricity
Percentage Range: 95-98

Emission Activity: ASIC miner electricity consumption
        Scope: Scope2
        Category: PurchasedElectricity

        Data Collection Requirements:
        - What to collect: Electricity usage
        - Unit of measure: kWh
 

In [37]:
print("\n" + "=" * 80)
print("QUERY 3: Find high-impact activities (>1% of emissions)")
print("=" * 80)
results = emission_db.query(
    query_texts=["electricity power consumption tracking"],
    n_results=10,
    where={"percentage_min": {"$gte": 1.0}}
)
for i, doc in enumerate(results["documents"][0]):
    print(f"\n--- Result {i+1} ---")
    print(f"Unit: {results['metadatas'][0][i]['unit_of_measure']}")
    print(f"Data to collect: {results['metadatas'][0][i]['data_to_collect']}")
    print(f"Impact: {results['metadatas'][0][i]['percentage_range']}%")
    print(f"\n{doc}")


QUERY 3: Find high-impact activities (>1% of emissions)

--- Result 1 ---
Unit: kwh
Data to collect: electricity usage
Impact: 95-98%

Emission Activity: ASIC miner electricity consumption
        Scope: Scope2
        Category: PurchasedElectricity

        Data Collection Requirements:
        - What to collect: Electricity usage
        - Unit of measure: kWh
        - Typical percentage of total footprint: 95-98%
        - Priority level: CRITICAL

        This is a Scope2 emission source in the PurchasedElectricity category.
        For Bitcoin mining operations, this activity typically represents 95-98% of total emissions.
        Collection priority: CRITICAL.

--- Result 2 ---
Unit: kwh
Data to collect: electricity usage
Impact: 1-3%

Emission Activity: Cooling system electricity
        Scope: Scope2
        Category: PurchasedElectricity

        Data Collection Requirements:
        - What to collect: Electricity usage
        - Unit of measure: kWh
        - Typical percen

In [38]:
print("\n" + "=" * 80)
print("QUERY 4: Find refrigerant and fugitive emissions")
print("=" * 80)
results = emission_db.query(
    query_texts=["refrigerant cooling system leaks"],
    n_results=3,
    where={"activity_category": "fugitiveemissions"}
)
for i, doc in enumerate(results["documents"][0]):
    print(f"\n--- Result {i+1} ---")
    print(f"Source: {results['metadatas'][0][i]['emission_source']}")
    print(f"Percentage Range: {results['metadatas'][0][i]['percentage_range']}")
    print(f"\n{doc}")


QUERY 4: Find refrigerant and fugitive emissions

--- Result 1 ---
Source: refrigerant leaks from immersion cooling
Percentage Range: 0.01-0.1

Emission Activity: Refrigerant leaks from immersion cooling
        Scope: Scope1
        Category: FugitiveEmissions

        Data Collection Requirements:
        - What to collect: Refrigerant quantity by type
        - Unit of measure: kg
        - Typical percentage of total footprint: 0.01-0.1%
        - Priority level: LOW

        This is a Scope1 emission source in the FugitiveEmissions category.
        For Bitcoin mining operations, this activity typically represents 0.01-0.1% of total emissions.
        Collection priority: LOW.

--- Result 2 ---
Source: refrigerant leaks from hvac systems
Percentage Range: 0.01-0.1

Emission Activity: Refrigerant leaks from HVAC systems
        Scope: Scope1
        Category: FugitiveEmissions

        Data Collection Requirements:
        - What to collect: Refrigerant quantity by type
        - 

In [39]:
print("\n" + "=" * 80)
print("QUERY 5: Find all Scope 1 activities by priority")
print("=" * 80)
results = emission_db.query(
    query_texts=["direct emissions from operations"],
    n_results=10,
    where={"emission_scope": "scope1"}
)
for i, doc in enumerate(results["documents"][0]):
    print(f"\n--- Scope 1 Activity {i+1} ---")
    print(f"Category: {results['metadatas'][0][i]['activity_category']}")
    print(f"Priority: {results['metadatas'][0][i]['priority_level']}")
    print(f"Percentage Range: {results['metadatas'][0][i]['percentage_range']}")


QUERY 5: Find all Scope 1 activities by priority

--- Scope 1 Activity 1 ---
Category: stationarycombustion
Priority: low
Percentage Range: 0.05-0.3

--- Scope 1 Activity 2 ---
Category: fugitiveemissions
Priority: low
Percentage Range: 0.01-0.1

--- Scope 1 Activity 3 ---
Category: mobilecombustion
Priority: low
Percentage Range: 0.01-0.05

--- Scope 1 Activity 4 ---
Category: stationarycombustion
Priority: low
Percentage Range: 0.1-0.5

--- Scope 1 Activity 5 ---
Category: stationarycombustion
Priority: low-medium
Percentage Range: 0.1-2

--- Scope 1 Activity 6 ---
Category: fugitiveemissions
Priority: low
Percentage Range: 0.01-0.1

--- Scope 1 Activity 7 ---
Category: mobilecombustion
Priority: low
Percentage Range: 0.01-0.05

--- Scope 1 Activity 8 ---
Category: fugitiveemissions
Priority: low
Percentage Range: 0.01-0.1

--- Scope 1 Activity 9 ---
Category: stationarycombustion
Priority: low-medium
Percentage Range: 0.1-1

--- Scope 1 Activity 10 ---
Category: fugitiveemissions
P

In [8]:
# Helper functions for advanced queries
def get_activities_by_priority(collection, min_priority_rank: int = 4):
    """
    Get all activities with priority rank >= min_priority_rank
    5 = critical, 4 = high, 3 = medium, 2 = low-medium, 1 = low
    """
    results = collection.query(
        query_texts=["emission activities"],
        n_results=100,
        where={"priority_rank": {"$gte": min_priority_rank}}
    )
    return results

def get_high_impact_activities(collection, min_percentage: float = 1.0):
    """Get activities that represent at least min_percentage of total emissions"""
    results = collection.query(
        query_texts=["high impact emission sources"],
        n_results=50,
        where={"percentage_min": {"$gte": min_percentage}}
    )
    return results

def get_scope2_critical_activities(collection):
    """Get only Scope 2 critical activities"""
    results = collection.query(
        query_texts=["electricity consumption tracking"],
        n_results=50,
        where={
            "$and": [
                {"emission_scope": "scope2"},
                {"priority_level": "critical"}
            ]
        }
    )
    return results


def get_activities_by_unit(collection, unit_type: str = "kwh"):
    """Get all activities measured in a specific unit"""
    results = collection.query(
        query_texts=[f"activities measured in {unit_type}"],
        n_results=50,
        where={"unit_of_measure": {"$contains": unit_type.lower()}}
    )
    return results

In [13]:
# Example usage of helper functions    
# print("\n" + "=" * 80)
print("ADVANCED QUERY: High priority activities (rank >= 4)")
print("=" * 80)
results = get_activities_by_priority(emission_db, min_priority_rank=4)
print(f"Found {len(results['documents'][0])} high-priority activities")
for i in range(min(5, len(results['documents'][0]))):
    print(f"\n{i+1}. {results['metadatas'][0][i]['emission_source']}")
    print(f"   Priority: {results['metadatas'][0][i]['priority_level']}")
    print(f"   Impact: {results['metadatas'][0][i]['percentage_range']}%")

ADVANCED QUERY: High priority activities (rank >= 4)
Found 8 high-priority activities

1. facility operations electricity
   Priority: high
   Impact: 0.1-0.5%

2. asic miner electricity consumption
   Priority: critical
   Impact: 95-98%

3. cooling system electricity
   Priority: critical
   Impact: 1-3%

4. power usage effectiveness
   Priority: critical
   Impact: nan%

5. grid carbon intensity
   Priority: critical
   Impact: nan%
